In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 0: [COLAB ONLY] Install GPU libraries (cuML for GPU Random Forest)
# ══════════════════════════════════════════════════════════════════════════════
# Run this cell ONLY on Google Colab with GPU runtime.
# Skip on local machine (will fall back to CPU sklearn).
#
# Runtime → Change runtime type → T4 GPU (or better)
# ══════════════════════════════════════════════════════════════════════════════

import subprocess, sys

def is_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if is_colab():
    print('🚀 Colab detected — installing cuML for GPU acceleration...')
    print('   (This takes ~2-3 minutes, but training will be 10-50x faster)')
    
    # Install RAPIDS cuML (GPU-accelerated ML)
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--quiet',
        'cudf-cu12', 'cuml-cu12',
        '--extra-index-url=https://pypi.nvidia.com'
    ])
    print('✅ cuML installed successfully!')
else:
    print('💻 Local machine detected — skipping cuML install (will use CPU sklearn)')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 1: Imports & Setup (with GPU detection)
# ══════════════════════════════════════════════════════════════════════════════
import os, sys, glob, gc, time, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report, confusion_matrix,
    mean_squared_error, mean_absolute_error, r2_score,
)

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 100, 'figure.figsize': (14, 6), 'font.size': 11})
sns.set_style('whitegrid')

# ── GPU / cuML Detection ──────────────────────────────────────────────────────
# cuML provides GPU-accelerated Random Forests (10-50x faster on Colab GPU)
# Install on Colab: !pip install cudf-cu12 cuml-cu12 --extra-index-url=https://pypi.nvidia.com

USE_GPU = False
try:
    import cuml
    from cuml.ensemble import RandomForestClassifier as cuRFC
    from cuml.ensemble import RandomForestRegressor as cuRFR
    USE_GPU = True
    print('✓ cuML detected — using GPU-accelerated Random Forest')
except ImportError:
    from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
    print('✗ cuML not found — using CPU sklearn (install cuML for 10-50x speedup on Colab GPU)')

# ── Project root ──────────────────────────────────────────────────────────────
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

DATA_DIR       = os.path.join(PROJECT_ROOT, 'data', 'processed')
WEIGHTS_DIR    = os.path.join(PROJECT_ROOT, 'model_weights')
RESULTS_DIR    = os.path.join(PROJECT_ROOT, 'results')
os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Horizons (same as src/features/labels.py DEFAULT_HORIZONS) ────────────────
HORIZONS = [10, 20, 50, 100]

# ── Discover tickers ──────────────────────────────────────────────────────────
tickers = sorted([
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
    and glob.glob(os.path.join(DATA_DIR, d, '*.parquet'))
])

print(f'Project root : {PROJECT_ROOT}')
print(f'Data dir     : {DATA_DIR}')
print(f'Tickers      : {tickers}')
print(f'Horizons     : {HORIZONS}')
print(f'Weights dir  : {WEIGHTS_DIR}')
print(f'USE_GPU      : {USE_GPU}')

Project root : /home/illionar/Projects/multi-horizon-ofi
Data dir     : /home/illionar/Projects/multi-horizon-ofi/data/processed
Tickers      : ['AAPL', 'AMZN', 'GOOG', 'MSFT', 'TSLA']
Horizons     : [10, 20, 50, 100]
Weights dir  : /home/illionar/Projects/multi-horizon-ofi/model_weights


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 2: Feature Engineering & Data Pipeline (fully self-contained)
# ══════════════════════════════════════════════════════════════════════════════
# All feature / label logic is inlined here — no src/ imports needed.
# Upload only this notebook + data/processed/ to Colab.

# ─────────────────────────────────────────────────────────────────────────────
# OFI (Order Flow Imbalance)
# Refs: Cont, Kukanov & Stoikov (2014), Xu, Gould & Howison (2019)
# ─────────────────────────────────────────────────────────────────────────────

def _level_ofi_arrays(bp, bs, ap, as_):
    """Vectorised single-level OFI on raw numpy arrays."""
    n = len(bp)
    ofi = np.zeros(n, dtype=np.float64)
    bp = bp.astype(np.float64)
    bs = bs.astype(np.float64)
    ap = ap.astype(np.float64)
    as_ = as_.astype(np.float64)

    delta_bid = np.where(
        bp[1:] > bp[:-1], bs[1:],
        np.where(bp[1:] < bp[:-1], -bs[:-1], bs[1:] - bs[:-1]),
    )
    delta_ask = np.where(
        ap[1:] < ap[:-1], -as_[1:],
        np.where(ap[1:] > ap[:-1], as_[:-1], as_[1:] - as_[:-1]),
    )
    ofi[1:] = delta_bid - delta_ask
    return ofi


def compute_multi_level_ofi(df, max_level=5):
    """
    Multi-level OFI for levels 1..max_level.
    Returns ofi_1..ofi_{max_level} and ofi_cumul_1..ofi_cumul_{max_level}.
    """
    result = {}
    cumul = np.zeros(len(df), dtype=np.float64)
    for lvl in range(1, max_level + 1):
        bp = df[f'bid_price_{lvl}'].values
        bs = df[f'bid_size_{lvl}'].values
        ap = df[f'ask_price_{lvl}'].values
        as_ = df[f'ask_size_{lvl}'].values
        ofi_lvl = _level_ofi_arrays(bp, bs, ap, as_)
        result[f'ofi_{lvl}'] = ofi_lvl
        cumul = cumul + ofi_lvl
        result[f'ofi_cumul_{lvl}'] = cumul.copy()
    return pd.DataFrame(result, index=df.index)


# ─────────────────────────────────────────────────────────────────────────────
# Microstructure features
# ─────────────────────────────────────────────────────────────────────────────

def compute_all_features(df, max_level=5):
    """
    Compute all microstructure features:
    mid_price, spread, volume_imbalance, weighted_mid_price,
    bid_depth, ask_depth, total_depth, depth_imbalance.
    """
    mid = ((df['ask_price_1'] + df['bid_price_1']) / 2.0).rename('mid_price')
    spread = (df['ask_price_1'] - df['bid_price_1']).rename('spread')

    bid_s = df['bid_size_1'].astype(np.float64)
    ask_s = df['ask_size_1'].astype(np.float64)
    total_s = bid_s + ask_s
    vi = ((bid_s - ask_s) / total_s.replace(0, np.nan)).fillna(0.0).rename('volume_imbalance')
    wmp = (df['ask_price_1'] * bid_s + df['bid_price_1'] * ask_s) / total_s.replace(0, np.nan)
    wmp = wmp.fillna(mid).rename('weighted_mid_price')

    bid_depth = sum(df[f'bid_size_{i}'].astype(np.float64) for i in range(1, max_level + 1))
    ask_depth = sum(df[f'ask_size_{i}'].astype(np.float64) for i in range(1, max_level + 1))
    depth_df = pd.DataFrame({
        'bid_depth': bid_depth,
        'ask_depth': ask_depth,
        'total_depth': bid_depth + ask_depth,
    }, index=df.index)

    total_d = depth_df['total_depth'].astype(np.float64)
    di = ((depth_df['bid_depth'] - depth_df['ask_depth']).astype(np.float64)
          / total_d.replace(0, np.nan)).fillna(0.0).rename('depth_imbalance')

    return pd.concat([mid, spread, vi, wmp, depth_df, di], axis=1)


# ─────────────────────────────────────────────────────────────────────────────
# Label generation
# Ref: Zhang, Zohren & Roberts (2019) — smoothed labelling
# ─────────────────────────────────────────────────────────────────────────────

def _mid_price_array(df):
    return (df['ask_price_1'].values + df['bid_price_1'].values) / 2.0


def _smoothed_mid(m, k):
    cumsum = np.cumsum(m)
    smoothed = np.full_like(m, np.nan)
    if k < len(m):
        smoothed[:len(m) - k] = (cumsum[k:] - cumsum[:-k]) / k
    return smoothed


def make_regression_labels(df, horizons=None):
    if horizons is None:
        horizons = HORIZONS
    m = _mid_price_array(df)
    result = {}
    for k in horizons:
        shifted = np.full_like(m, np.nan)
        if k < len(m):
            shifted[:len(m) - k] = m[k:]
        result[f'delta_mid_{k}'] = shifted - m
    return pd.DataFrame(result, index=df.index)


def make_classification_labels(df, horizons=None, alpha=0.5, use_smoothing=True):
    if horizons is None:
        horizons = HORIZONS
    m = _mid_price_array(df)
    result = {}
    for k in horizons:
        if use_smoothing:
            future_m = _smoothed_mid(m, k)
        else:
            future_m = np.full_like(m, np.nan)
            if k < len(m):
                future_m[:len(m) - k] = m[k:]
        pct_change = (future_m - m) / m
        valid = pct_change[~np.isnan(pct_change)]
        sigma = np.std(valid)
        threshold = alpha * sigma
        labels = np.full(len(m), np.nan)
        labels[pct_change > threshold] = 2
        labels[pct_change < -threshold] = 0
        mask = (~np.isnan(pct_change)) & np.isnan(labels)
        labels[mask] = 1
        result[f'label_{k}'] = labels
    return pd.DataFrame(result, index=df.index)


# ─────────────────────────────────────────────────────────────────────────────
# Data loading pipeline (memory-optimised)
# ─────────────────────────────────────────────────────────────────────────────

# The ONLY 20 parquet columns we ever touch — everything else is wasted RAM.
_LOB_COLUMNS = []
for _lvl in range(1, 6):
    _LOB_COLUMNS.extend([
        f'ask_price_{_lvl}', f'ask_size_{_lvl}',
        f'bid_price_{_lvl}', f'bid_size_{_lvl}',
    ])


def build_features_and_labels_from_parquet(
    parquet_path, ofi_levels=5, horizons=None, alpha=0.5,
):
    """
    Load one day's parquet → compute features + labels → return numpy arrays.

    Memory-optimised:
      • Reads only the 20 LOB columns (not all 46+ in the parquet)
      • Intermediate DataFrames are deleted as soon as possible
    """
    if horizons is None:
        horizons = HORIZONS

    # ── Read ONLY the columns we need (halves DataFrame memory) ──────────
    df = pd.read_parquet(parquet_path, columns=_LOB_COLUMNS)

    # OFI features  (10 float64 cols)
    ofi_df = compute_multi_level_ofi(df, max_level=ofi_levels)

    # Microstructure features  (8 float64 cols)
    micro_df = compute_all_features(df, max_level=ofi_levels)

    # Combine into single feature frame  (38 cols: 10 ofi + 8 micro + 20 LOB)
    features = pd.concat([ofi_df, micro_df, df], axis=1)
    feat_names = list(features.columns)
    del ofi_df, micro_df, df
    gc.collect()

    # Labels  (we can reconstruct mid-price from the features frame)
    mid_vals = (features['ask_price_1'].values + features['bid_price_1'].values) / 2.0

    # Regression labels
    reg_dict = {}
    for k in horizons:
        shifted = np.full_like(mid_vals, np.nan)
        if k < len(mid_vals):
            shifted[:len(mid_vals) - k] = mid_vals[k:]
        reg_dict[f'delta_mid_{k}'] = shifted - mid_vals

    # Classification labels
    cls_dict = {}
    for k in horizons:
        cumsum = np.cumsum(mid_vals)
        smoothed = np.full_like(mid_vals, np.nan)
        if k < len(mid_vals):
            smoothed[:len(mid_vals) - k] = (cumsum[k:] - cumsum[:-k]) / k
        pct_change = (smoothed - mid_vals) / mid_vals
        valid = pct_change[~np.isnan(pct_change)]
        sigma = np.std(valid) if len(valid) > 0 else 1e-9
        threshold = alpha * sigma
        labels = np.full(len(mid_vals), np.nan)
        labels[pct_change > threshold] = 2
        labels[pct_change < -threshold] = 0
        mask = (~np.isnan(pct_change)) & np.isnan(labels)
        labels[mask] = 1
        cls_dict[f'label_{k}'] = labels

    del mid_vals

    # Trim NaN tail
    max_h = max(horizons)
    valid_end = len(features) - max_h
    if valid_end <= 0:
        raise ValueError(f'Day has {len(features)} events but max horizon is {max_h}')

    # Convert to float32 numpy & free everything else
    X     = features.values[:valid_end].astype(np.float32)
    del features
    gc.collect()

    y_reg = np.column_stack([reg_dict[f'delta_mid_{k}'][:valid_end] for k in horizons]).astype(np.float32)
    y_cls = np.column_stack([cls_dict[f'label_{k}'][:valid_end] for k in horizons]).astype(np.float32)
    del reg_dict, cls_dict
    gc.collect()

    return X, y_reg, y_cls, feat_names


def load_ticker_data(ticker, data_dir=DATA_DIR, horizons=None, max_days=None, subsample=1):
    """
    Load all (or max_days) parquet files for a ticker.
    Optional subsampling: keep every `subsample`-th row per day.
    """
    if horizons is None:
        horizons = HORIZONS

    ticker_dir = os.path.join(data_dir, ticker)
    parquet_files = sorted(glob.glob(os.path.join(ticker_dir, '*.parquet')))
    if max_days is not None:
        parquet_files = parquet_files[:max_days]
    if not parquet_files:
        raise FileNotFoundError(f'No parquet files in {ticker_dir}')

    all_X, all_yr, all_yc = [], [], []
    feat_names = None

    for pf in parquet_files:
        X, yr, yc, fnames = build_features_and_labels_from_parquet(pf, horizons=horizons)
        if subsample > 1:
            X  = X[::subsample]
            yr = yr[::subsample]
            yc = yc[::subsample]
        all_X.append(X)
        all_yr.append(yr)
        all_yc.append(yc)
        if feat_names is None:
            feat_names = fnames

    X     = np.concatenate(all_X, axis=0)
    y_reg = np.concatenate(all_yr, axis=0)
    y_cls = np.concatenate(all_yc, axis=0)
    del all_X, all_yr, all_yc
    gc.collect()

    mask = ~(np.isnan(X).any(axis=1) | np.isnan(y_reg).any(axis=1) | np.isnan(y_cls).any(axis=1))
    X, y_reg, y_cls = X[mask], y_reg[mask], y_cls[mask]

    return X, y_reg, y_cls, feat_names


def temporal_split(X, y_reg, y_cls, train_frac=0.6, val_frac=0.2):
    n = len(X)
    t1 = int(n * train_frac)
    t2 = int(n * (train_frac + val_frac))
    return (X[:t1], y_reg[:t1], y_cls[:t1]), \
           (X[t1:t2], y_reg[t1:t2], y_cls[t1:t2]), \
           (X[t2:], y_reg[t2:], y_cls[t2:])


print('✓ All feature engineering & data pipeline functions defined (self-contained).')

✓ Data pipeline functions defined.


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 3: Training & Evaluation Helper Functions (GPU/CPU adaptive)
# ══════════════════════════════════════════════════════════════════════════════

def train_rf_classifier(
    X_train, y_cls_train, X_val, y_cls_val,
    horizons=HORIZONS,
    n_estimators=200,
    max_depth=16,
    min_samples_leaf=20,
    n_jobs=-1,
    random_state=42,
):
    """
    Train one RF Classifier per horizon.
    Uses cuML (GPU) if available, else sklearn (CPU).
    Returns dict: {horizon: fitted_model}
    """
    models = {}
    
    for i, h in enumerate(horizons):
        y_tr = y_cls_train[:, i].astype(np.int32)
        y_va = y_cls_val[:, i].astype(np.int32)
        
        if USE_GPU:
            # cuML GPU Random Forest
            clf = cuRFC(
                n_estimators=n_estimators,
                max_depth=max_depth,
                min_samples_leaf=min_samples_leaf,
                n_streams=4,              # parallel GPU streams
                random_state=random_state,
            )
            backend = 'GPU'
        else:
            # sklearn CPU Random Forest
            clf = RandomForestClassifier(
                n_estimators=n_estimators,
                max_depth=max_depth,
                min_samples_leaf=min_samples_leaf,
                class_weight='balanced',
                n_jobs=n_jobs,
                random_state=random_state,
            )
            backend = 'CPU'
        
        t0 = time.time()
        clf.fit(X_train, y_tr)
        train_time = time.time() - t0
        
        # Validation score
        val_pred = clf.predict(X_val)
        if USE_GPU:
            val_pred = val_pred.get() if hasattr(val_pred, 'get') else np.array(val_pred)
        val_f1 = f1_score(y_va, val_pred, average='macro', zero_division=0)
        val_acc = accuracy_score(y_va, val_pred)
        
        models[h] = clf
        print(f'  h={h:>3d}  |  Val Acc={val_acc:.4f}  F1={val_f1:.4f}  |  '
              f'{train_time:.1f}s ({backend})')
    
    return models


def train_rf_regressor(
    X_train, y_reg_train, X_val, y_reg_val,
    horizons=HORIZONS,
    n_estimators=200,
    max_depth=16,
    min_samples_leaf=20,
    n_jobs=-1,
    random_state=42,
):
    """
    Train one RF Regressor per horizon.
    Uses cuML (GPU) if available, else sklearn (CPU).
    Returns dict: {horizon: fitted_model}
    """
    models = {}
    
    for i, h in enumerate(horizons):
        y_tr = y_reg_train[:, i].astype(np.float32)
        y_va = y_reg_val[:, i].astype(np.float32)
        
        if USE_GPU:
            # cuML GPU Random Forest
            reg = cuRFR(
                n_estimators=n_estimators,
                max_depth=max_depth,
                min_samples_leaf=min_samples_leaf,
                n_streams=4,
                random_state=random_state,
            )
            backend = 'GPU'
        else:
            # sklearn CPU Random Forest
            reg = RandomForestRegressor(
                n_estimators=n_estimators,
                max_depth=max_depth,
                min_samples_leaf=min_samples_leaf,
                n_jobs=n_jobs,
                random_state=random_state,
            )
            backend = 'CPU'
        
        t0 = time.time()
        reg.fit(X_train, y_tr)
        train_time = time.time() - t0
        
        # Validation score
        val_pred = reg.predict(X_val)
        if USE_GPU:
            val_pred = val_pred.get() if hasattr(val_pred, 'get') else np.array(val_pred)
        val_r2  = r2_score(y_va, val_pred)
        val_mae = mean_absolute_error(y_va, val_pred)
        
        models[h] = reg
        print(f'  h={h:>3d}  |  Val R²={val_r2:.4f}  MAE={val_mae:.6f}  |  '
              f'{train_time:.1f}s ({backend})')
    
    return models


def evaluate_classifier(models, X_test, y_cls_test, horizons=HORIZONS):
    """Evaluate RF classifiers on test set. Returns metrics dict."""
    metrics = {}
    for i, h in enumerate(horizons):
        y_true = y_cls_test[:, i].astype(int)
        y_pred = models[h].predict(X_test)
        if USE_GPU:
            y_pred = y_pred.get() if hasattr(y_pred, 'get') else np.array(y_pred)
        
        metrics[f'h{h}_accuracy']        = accuracy_score(y_true, y_pred)
        metrics[f'h{h}_f1_macro']        = f1_score(y_true, y_pred, average='macro', zero_division=0)
        metrics[f'h{h}_f1_weighted']     = f1_score(y_true, y_pred, average='weighted', zero_division=0)
        metrics[f'h{h}_precision_macro'] = precision_score(y_true, y_pred, average='macro', zero_division=0)
        metrics[f'h{h}_recall_macro']    = recall_score(y_true, y_pred, average='macro', zero_division=0)
    
    return metrics


def evaluate_regressor(models, X_test, y_reg_test, horizons=HORIZONS):
    """Evaluate RF regressors on test set. Returns metrics dict."""
    metrics = {}
    for i, h in enumerate(horizons):
        y_true = y_reg_test[:, i]
        y_pred = models[h].predict(X_test)
        if USE_GPU:
            y_pred = y_pred.get() if hasattr(y_pred, 'get') else np.array(y_pred)
        
        metrics[f'h{h}_mse']  = float(mean_squared_error(y_true, y_pred))
        metrics[f'h{h}_rmse'] = float(np.sqrt(mean_squared_error(y_true, y_pred)))
        metrics[f'h{h}_mae']  = float(mean_absolute_error(y_true, y_pred))
        metrics[f'h{h}_r2']   = float(r2_score(y_true, y_pred))
        
        # Information Coefficient (Pearson corr between pred and actual)
        if np.std(y_true) > 1e-12 and np.std(y_pred) > 1e-12:
            metrics[f'h{h}_ic'] = float(np.corrcoef(y_true, y_pred)[0, 1])
        else:
            metrics[f'h{h}_ic'] = 0.0
    
    return metrics


def print_clf_metrics(metrics, horizons=HORIZONS):
    """Pretty-print classifier metrics."""
    print(f'{"Horizon":>8s} {"Accuracy":>10s} {"F1-macro":>10s} {"F1-wt":>10s} {"Precision":>10s} {"Recall":>10s}')
    print('─' * 62)
    for h in horizons:
        print(f'  h={h:<4d} '
              f'{metrics[f"h{h}_accuracy"]:10.4f} '
              f'{metrics[f"h{h}_f1_macro"]:10.4f} '
              f'{metrics[f"h{h}_f1_weighted"]:10.4f} '
              f'{metrics[f"h{h}_precision_macro"]:10.4f} '
              f'{metrics[f"h{h}_recall_macro"]:10.4f}')


def print_reg_metrics(metrics, horizons=HORIZONS):
    """Pretty-print regressor metrics."""
    print(f'{"Horizon":>8s} {"MSE":>12s} {"RMSE":>10s} {"MAE":>10s} {"R²":>10s} {"IC":>10s}')
    print('─' * 66)
    for h in horizons:
        print(f'  h={h:<4d} '
              f'{metrics[f"h{h}_mse"]:12.8f} '
              f'{metrics[f"h{h}_rmse"]:10.6f} '
              f'{metrics[f"h{h}_mae"]:10.6f} '
              f'{metrics[f"h{h}_r2"]:10.4f} '
              f'{metrics[f"h{h}_ic"]:10.4f}')


print(f'✓ Training & evaluation functions defined (USE_GPU={USE_GPU}).')

✓ Training & evaluation functions defined.


---
## ⚡ TRIAL RUN — Error Check (Tiny Data)

Runs the **entire pipeline end-to-end** on ~500 rows to verify:
- Data loading works
- Feature engineering produces correct shapes
- Both RF models train without errors
- Evaluation & saving works

**Run this cell on your laptop first.** If it passes, the full-scale cell below will work on Colab.

In [10]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 4: ⚡ TRIAL RUN — Validate Pipeline (Tiny Data, ~30 seconds)
# ══════════════════════════════════════════════════════════════════════════════

print('=' * 70)
print('TRIAL RUN — Error checking on tiny data subset')
print('=' * 70)

TRIAL_TICKER = tickers[0]  # First available ticker

# ── 1. Load just 1 day, heavily subsampled ────────────────────────────────────
print(f'\n[1/6] Loading 1 day of {TRIAL_TICKER} (subsampled 1/100)...')
X_trial, y_reg_trial, y_cls_trial, feat_names = load_ticker_data(
    TRIAL_TICKER, max_days=1, subsample=100
)
print(f'  X shape     : {X_trial.shape}')
print(f'  y_reg shape : {y_reg_trial.shape}')
print(f'  y_cls shape : {y_cls_trial.shape}')
print(f'  Features    : {len(feat_names)}')
print(f'  Feature list: {feat_names}')

# ── 2. Temporal split ─────────────────────────────────────────────────────────
print(f'\n[2/6] Temporal split (60/20/20)...')
train_t, val_t, test_t = temporal_split(X_trial, y_reg_trial, y_cls_trial)
X_tr, yr_tr, yc_tr = train_t
X_va, yr_va, yc_va = val_t
X_te, yr_te, yc_te = test_t
print(f'  Train: {X_tr.shape[0]:>6,}  Val: {X_va.shape[0]:>6,}  Test: {X_te.shape[0]:>6,}')

# ── 3. Scale features ─────────────────────────────────────────────────────────
print(f'\n[3/6] Scaling features (StandardScaler fit on train)...')
scaler_trial = StandardScaler()
X_tr_s = scaler_trial.fit_transform(X_tr)
X_va_s = scaler_trial.transform(X_va)
X_te_s = scaler_trial.transform(X_te)
print(f'  Scaler fitted. Mean range: [{X_tr_s.mean(axis=0).min():.2f}, {X_tr_s.mean(axis=0).max():.2f}]')

# ── 4. Train RF Classifier (tiny) ─────────────────────────────────────────────
print(f'\n[4/6] Training RF Classifier (n_estimators=5, max_depth=4)...')
clf_models_trial = train_rf_classifier(
    X_tr_s, yc_tr, X_va_s, yc_va,
    horizons=HORIZONS,
    n_estimators=5,     # tiny for speed
    max_depth=4,        # shallow for speed
    min_samples_leaf=2,
)

# ── 5. Train RF Regressor (tiny) ──────────────────────────────────────────────
print(f'\n[5/6] Training RF Regressor (n_estimators=5, max_depth=4)...')
reg_models_trial = train_rf_regressor(
    X_tr_s, yr_tr, X_va_s, yr_va,
    horizons=HORIZONS,
    n_estimators=5,
    max_depth=4,
    min_samples_leaf=2,
)

# ── 6. Evaluate on test set ───────────────────────────────────────────────────
print(f'\n[6/6] Evaluating on test set...')

clf_metrics_trial = evaluate_classifier(clf_models_trial, X_te_s, yc_te)
reg_metrics_trial = evaluate_regressor(reg_models_trial, X_te_s, yr_te)

print(f'\n── RF Classifier (Trial) ──')
print_clf_metrics(clf_metrics_trial)

print(f'\n── RF Regressor (Trial) ──')
print_reg_metrics(reg_metrics_trial)

# ── 7. Test saving ────────────────────────────────────────────────────────────
trial_save_path = os.path.join(WEIGHTS_DIR, '_trial_test_rf.joblib')
joblib.dump({'test': 'ok'}, trial_save_path)
_ = joblib.load(trial_save_path)
os.remove(trial_save_path)
print(f'\n✅ Save/load test passed.')

# ── Cleanup ───────────────────────────────────────────────────────────────────
del X_trial, y_reg_trial, y_cls_trial
del train_t, val_t, test_t
del X_tr, yr_tr, yc_tr, X_va, yr_va, yc_va, X_te, yr_te, yc_te
del X_tr_s, X_va_s, X_te_s
del clf_models_trial, reg_models_trial
gc.collect()

print('\n' + '=' * 70)
print('✅ TRIAL RUN PASSED — No errors. Safe to proceed to full training.')
print('=' * 70)

TRIAL RUN — Error checking on tiny data subset

[1/6] Loading 1 day of AAPL (subsampled 1/100)...
  X shape     : (18641, 38)
  y_reg shape : (18641, 4)
  y_cls shape : (18641, 4)
  Features    : 38
  Feature list: ['ofi_1', 'ofi_cumul_1', 'ofi_2', 'ofi_cumul_2', 'ofi_3', 'ofi_cumul_3', 'ofi_4', 'ofi_cumul_4', 'ofi_5', 'ofi_cumul_5', 'mid_price', 'spread', 'volume_imbalance', 'weighted_mid_price', 'bid_depth', 'ask_depth', 'total_depth', 'depth_imbalance', 'ask_price_1', 'ask_size_1', 'bid_price_1', 'bid_size_1', 'ask_price_2', 'ask_size_2', 'bid_price_2', 'bid_size_2', 'ask_price_3', 'ask_size_3', 'bid_price_3', 'bid_size_3', 'ask_price_4', 'ask_size_4', 'bid_price_4', 'bid_size_4', 'ask_price_5', 'ask_size_5', 'bid_price_5', 'bid_size_5']

[2/6] Temporal split (60/20/20)...
  Train: 11,184  Val:  3,728  Test:  3,729

[3/6] Scaling features (StandardScaler fit on train)...
  Scaler fitted. Mean range: [-0.00, 0.00]

[4/6] Training RF Classifier (n_estimators=5, max_depth=4)...
  h= 10

---
## 🚀 FULL-SCALE TRAINING

Loads **all tickers × all days**, trains RF models with production hyperparameters.

**Run this on Google Colab** (or any machine with ≥16 GB RAM).

Estimated time: 5–15 minutes depending on data size and CPU cores.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 5: 🚀 FULL-SCALE — Disk-Backed Streaming Load (with cache)
# ══════════════════════════════════════════════════════════════════════════════
#
# If .npy files already exist on disk from a previous run → skip processing,
# just memory-map them instantly. Delete TEMP_DIR to force recompute.
#
# Tuning knobs:
#   SUBSAMPLE        — keep every N-th row (higher = less data, less RAM)
#   TEMP_DIR         — where to store temp files (default: data/temp_rf/)
#   FORCE_RECOMPUTE  — set True to ignore cache and reprocess everything
# ══════════════════════════════════════════════════════════════════════════════
import psutil, traceback, shutil
import pyarrow.parquet as pq

# ── Tuning knobs ─────────────────────────────────────────────────────────────
SUBSAMPLE        = 10
TEMP_DIR         = os.path.join(PROJECT_ROOT, 'data', 'temp_rf')
FORCE_RECOMPUTE  = False    # set True to reprocess even if cache exists

# ── Helpers ──────────────────────────────────────────────────────────────────
BYTES_PER_ROW = 38 * 4 + 4 * 4 + 4 * 4   # 184 B
MAX_HORIZON   = max(HORIZONS)

def _mem_gb():
    return psutil.Process(os.getpid()).memory_info().rss / 1e9

def _avail_gb():
    return psutil.virtual_memory().available / 1e9

# ── Cache file paths ─────────────────────────────────────────────────────────
X_path    = os.path.join(TEMP_DIR, 'X.npy')
yr_path   = os.path.join(TEMP_DIR, 'y_reg.npy')
yc_path   = os.path.join(TEMP_DIR, 'y_cls.npy')
meta_path = os.path.join(TEMP_DIR, 'cache_meta.json')

cache_exists = (
    not FORCE_RECOMPUTE
    and os.path.isfile(X_path)
    and os.path.isfile(yr_path)
    and os.path.isfile(yc_path)
    and os.path.isfile(meta_path)
)

print('=' * 70)
print('FULL-SCALE TRAINING — Disk-backed streaming (minimal RAM)')
print('=' * 70)

if cache_exists:
    # ══════════════════════════════════════════════════════════════════════
    # FAST PATH: Load from cache
    # ══════════════════════════════════════════════════════════════════════
    print(f'✅ Cache found in {TEMP_DIR}/ — skipping feature computation!')
    
    with open(meta_path, 'r') as f:
        cache_meta = json.load(f)
    
    feat_names = cache_meta['feat_names']
    
    print(f'  Cached rows    : {cache_meta["total_rows"]:,}')
    print(f'  Cached subsample: {cache_meta["subsample"]}')
    print(f'  Cached tickers : {cache_meta["tickers"]}')
    print(f'  Features ({len(feat_names)})')
    
    t0_load = time.time()

else:
    # ══════════════════════════════════════════════════════════════════════
    # FULL PATH: Process parquets → stream to disk
    # ══════════════════════════════════════════════════════════════════════
    print(f'[CONFIG] SUBSAMPLE = {SUBSAMPLE}  (keep every {SUBSAMPLE}-th row)')
    print(f'[CONFIG] TEMP_DIR  = {TEMP_DIR}')
    print(f'[START]  RSS = {_mem_gb():.2f} GB  |  Available = {_avail_gb():.2f} GB')
    
    # Clean and recreate temp dir
    if os.path.exists(TEMP_DIR):
        shutil.rmtree(TEMP_DIR)
    os.makedirs(TEMP_DIR, exist_ok=True)
    
    # ── Count rows ───────────────────────────────────────────────────────
    print(f'\n[SCAN] Counting rows from parquet metadata...')
    
    ticker_files = {}
    total_rows_raw = 0
    
    for ticker in tickers:
        ticker_dir = os.path.join(DATA_DIR, ticker)
        pfiles = sorted(glob.glob(os.path.join(ticker_dir, '*.parquet')))
        ticker_files[ticker] = pfiles
        for pf in pfiles:
            total_rows_raw += max(0, pq.read_metadata(pf).num_rows - MAX_HORIZON)
    
    est_rows = total_rows_raw // SUBSAMPLE
    est_gb   = est_rows * BYTES_PER_ROW / 1e9
    print(f'  Raw rows   : {total_rows_raw:,}')
    print(f'  Subsampled : ~{est_rows:,} rows  ≈  {est_gb:.2f} GB on disk')
    
    # ── Stream to disk ───────────────────────────────────────────────────
    print(f'\n{"─"*70}')
    print(f'Streaming to disk (one day at a time)...')
    t0_load = time.time()
    
    X_fp  = open(X_path,  'wb')
    yr_fp = open(yr_path, 'wb')
    yc_fp = open(yc_path, 'wb')
    
    def write_npy_header(fp, shape, dtype):
        import struct
        header = {'descr': np.dtype(dtype).str, 'fortran_order': False, 'shape': shape}
        header_bytes = repr(header).encode('latin1')
        pad_len = 64 - ((10 + len(header_bytes)) % 64)
        header_bytes = header_bytes + b' ' * pad_len + b'\n'
        fp.write(b'\x93NUMPY\x01\x00')
        fp.write(struct.pack('<H', len(header_bytes)))
        fp.write(header_bytes)
        return fp.tell()
    
    write_npy_header(X_fp,  (0, 38), np.float32)
    write_npy_header(yr_fp, (0, 4),  np.float32)
    write_npy_header(yc_fp, (0, 4),  np.float32)
    
    total_rows_written = 0
    feat_names = None
    
    for t_idx, ticker in enumerate(tickers):
        pfiles = ticker_files[ticker]
        print(f'\n  [{t_idx+1}/{len(tickers)}] {ticker}  ({len(pfiles)} days)  '
              f'RSS={_mem_gb():.2f} GB  avail={_avail_gb():.2f} GB')
        
        for d_idx, pf in enumerate(pfiles):
            try:
                X, yr, yc, fnames = build_features_and_labels_from_parquet(
                    pf, horizons=HORIZONS)
                
                if feat_names is None:
                    feat_names = fnames
                
                if SUBSAMPLE > 1:
                    X  = X[::SUBSAMPLE]
                    yr = yr[::SUBSAMPLE]
                    yc = yc[::SUBSAMPLE]
                
                mask = ~(np.isnan(X).any(axis=1) | np.isnan(yr).any(axis=1) |
                         np.isnan(yc).any(axis=1))
                X, yr, yc = X[mask], yr[mask], yc[mask]
                
                X.tofile(X_fp)
                yr.tofile(yr_fp)
                yc.tofile(yc_fp)
                total_rows_written += len(X)
                
                del X, yr, yc, mask
                gc.collect()
                
            except Exception as e:
                print(f'    ⚠ {os.path.basename(pf)}: {e}')
                continue
        
        print(f'    → Written {total_rows_written:,} rows total  |  RSS={_mem_gb():.2f} GB')
    
    X_fp.close()
    yr_fp.close()
    yc_fp.close()
    
    # ── Fix headers ──────────────────────────────────────────────────────
    print(f'\n[FIX] Updating .npy headers with final row count: {total_rows_written:,}')
    
    def fix_npy_header(path, shape, dtype):
        import struct
        with open(path, 'r+b') as f:
            f.seek(8)
            header_len = struct.unpack('<H', f.read(2))[0]
            header = {'descr': np.dtype(dtype).str, 'fortran_order': False, 'shape': shape}
            header_bytes = repr(header).encode('latin1')
            pad_len = header_len - len(header_bytes) - 1
            if pad_len < 0:
                raise ValueError(f'Header too long! Need {len(header_bytes)+1}, have {header_len}')
            header_bytes = header_bytes + b' ' * pad_len + b'\n'
            f.seek(10)
            f.write(header_bytes)
    
    fix_npy_header(X_path,  (total_rows_written, 38), np.float32)
    fix_npy_header(yr_path, (total_rows_written, 4),  np.float32)
    fix_npy_header(yc_path, (total_rows_written, 4),  np.float32)
    
    # ── Save cache metadata ──────────────────────────────────────────────
    cache_meta = {
        'total_rows': total_rows_written,
        'subsample': SUBSAMPLE,
        'tickers': tickers,
        'horizons': HORIZONS,
        'feat_names': feat_names,
        'created': pd.Timestamp.now().isoformat(),
    }
    with open(meta_path, 'w') as f:
        json.dump(cache_meta, f, indent=2)
    print(f'  Cache metadata saved to {meta_path}')

# ══════════════════════════════════════════════════════════════════════════════
# Memory-map the files (shared by both paths)
# ══════════════════════════════════════════════════════════════════════════════
print(f'\n[MMAP] Memory-mapping arrays from disk...')

X_all     = np.load(X_path,  mmap_mode='r')
y_reg_all = np.load(yr_path, mmap_mode='r')
y_cls_all = np.load(yc_path, mmap_mode='r')

load_time = time.time() - t0_load

X_size_gb  = os.path.getsize(X_path) / 1e9
yr_size_mb = os.path.getsize(yr_path) / 1e6
yc_size_mb = os.path.getsize(yc_path) / 1e6

print(f'\n{"═"*70}')
print(f'✅ LOADING COMPLETE {"(from cache)" if cache_exists else "(freshly computed)"}')
print(f'  Total dataset : {X_all.shape[0]:>12,} rows × {X_all.shape[1]} features')
print(f'  Subsample     : {SUBSAMPLE}  (every {SUBSAMPLE}-th row)')
print(f'  Disk usage    : X={X_size_gb:.2f} GB  y_reg={yr_size_mb:.1f} MB  y_cls={yc_size_mb:.1f} MB')
print(f'  Process RSS   : {_mem_gb():.2f} GB  (arrays are memory-mapped, not in RAM!)')
print(f'  Available RAM : {_avail_gb():.2f} GB')
print(f'  Load time     : {load_time:.1f}s')
print(f'  Temp files    : {TEMP_DIR}/')
print(f'  Features ({len(feat_names)}): {feat_names}')
print(f'\n  💡 To force recompute: set FORCE_RECOMPUTE = True or delete {TEMP_DIR}/')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 6: Temporal Split & Scaling (save to disk, then memory-map)
# ══════════════════════════════════════════════════════════════════════════════
#
# Strategy: Scale data → save to disk → memory-map (no RAM usage)
# Training loop will load only what it needs into RAM one piece at a time.
# ══════════════════════════════════════════════════════════════════════════════

print('Temporal split (60% train / 20% val / 20% test)...')

n = len(X_all)
t1 = int(n * 0.6)
t2 = int(n * 0.8)

print(f'  Total : {n:>10,} rows (memory-mapped on disk)')
print(f'  Train : {t1:>10,} rows  (indices 0:{t1})')
print(f'  Val   : {t2-t1:>10,} rows  (indices {t1}:{t2})')
print(f'  Test  : {n-t2:>10,} rows  (indices {t2}:{n})')

# ── Load splits into RAM temporarily for scaling ─────────────────────────────
print(f'\n[SCALING] Loading data into RAM for StandardScaler fit...')
X_train     = np.array(X_all[:t1])
y_reg_train = np.array(y_reg_all[:t1])
y_cls_train = np.array(y_cls_all[:t1])

X_val     = np.array(X_all[t1:t2])
y_reg_val = np.array(y_reg_all[t1:t2])
y_cls_val = np.array(y_cls_all[t1:t2])

X_test     = np.array(X_all[t2:])
y_reg_test = np.array(y_reg_all[t2:])
y_cls_test = np.array(y_cls_all[t2:])

# Close memory-mapped files (we'll save processed data to new files)
del X_all, y_reg_all, y_cls_all
gc.collect()

print(f'  Data loaded. RSS={_mem_gb():.2f} GB')

# ── Scale features ───────────────────────────────────────────────────────────
print('\nFitting StandardScaler on train set...')
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)

del X_train, X_val, X_test
gc.collect()
print(f'  Scaler fitted. RSS={_mem_gb():.2f} GB')

# ── Save scaled arrays to disk ───────────────────────────────────────────────
print('\n[SAVE] Writing scaled arrays to disk (to free RAM)...')

scaled_dir = os.path.join(TEMP_DIR, 'scaled')
os.makedirs(scaled_dir, exist_ok=True)

X_train_path = os.path.join(scaled_dir, 'X_train_s.npy')
X_val_path   = os.path.join(scaled_dir, 'X_val_s.npy')
X_test_path  = os.path.join(scaled_dir, 'X_test_s.npy')

y_reg_train_path = os.path.join(scaled_dir, 'y_reg_train.npy')
y_reg_val_path   = os.path.join(scaled_dir, 'y_reg_val.npy')
y_reg_test_path  = os.path.join(scaled_dir, 'y_reg_test.npy')

y_cls_train_path = os.path.join(scaled_dir, 'y_cls_train.npy')
y_cls_val_path   = os.path.join(scaled_dir, 'y_cls_val.npy')
y_cls_test_path  = os.path.join(scaled_dir, 'y_cls_test.npy')

np.save(X_train_path, X_train_s)
np.save(X_val_path, X_val_s)
np.save(X_test_path, X_test_s)

np.save(y_reg_train_path, y_reg_train)
np.save(y_reg_val_path, y_reg_val)
np.save(y_reg_test_path, y_reg_test)

np.save(y_cls_train_path, y_cls_train)
np.save(y_cls_val_path, y_cls_val)
np.save(y_cls_test_path, y_cls_test)

print(f'  Saved 9 files to {scaled_dir}/')
print(f'  X_train: {os.path.getsize(X_train_path)/1e9:.2f} GB')
print(f'  X_val:   {os.path.getsize(X_val_path)/1e6:.1f} MB')
print(f'  X_test:  {os.path.getsize(X_test_path)/1e6:.1f} MB')

# ── Free RAM and memory-map ──────────────────────────────────────────────────
del X_train_s, X_val_s, X_test_s
del y_reg_train, y_reg_val, y_reg_test
del y_cls_train, y_cls_val, y_cls_test
gc.collect()

print(f'\n[MMAP] Memory-mapping scaled arrays (zero RAM usage)...')

X_train_s   = np.load(X_train_path, mmap_mode='r')
X_val_s     = np.load(X_val_path, mmap_mode='r')
X_test_s    = np.load(X_test_path, mmap_mode='r')

y_reg_train = np.load(y_reg_train_path, mmap_mode='r')
y_reg_val   = np.load(y_reg_val_path, mmap_mode='r')
y_reg_test  = np.load(y_reg_test_path, mmap_mode='r')

y_cls_train = np.load(y_cls_train_path, mmap_mode='r')
y_cls_val   = np.load(y_cls_val_path, mmap_mode='r')
y_cls_test  = np.load(y_cls_test_path, mmap_mode='r')

print(f'  All arrays memory-mapped. RSS={_mem_gb():.2f} GB  |  Avail={_avail_gb():.2f} GB')

# Label distribution
print('\nClassification label distribution (train):')
for i, h in enumerate(HORIZONS):
    counts = np.bincount(y_cls_train[:, i].astype(int), minlength=3)
    pcts = counts / counts.sum() * 100
    print(f'  h={h:>3d}  →  DOWN={pcts[0]:.1f}%  STAT={pcts[1]:.1f}%  UP={pcts[2]:.1f}%')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 7: 🌲 Train, Evaluate & Save (EXTREME Memory Efficiency)
# ══════════════════════════════════════════════════════════════════════════════
#
# Peak RAM optimization:
#   - Training data is memory-mapped (on disk, not RAM)
#   - Load ONLY when training: copy train split → train → free immediately
#   - Load ONLY when evaluating: copy test split → evaluate → free immediately
#   - Only ONE array in RAM at any time
# ══════════════════════════════════════════════════════════════════════════════

RF_PARAMS = {
    'n_estimators':    200,
    'max_depth':       16,
    'min_samples_leaf': 20,
    'n_jobs':          -1,
    'random_state':    42,
}

print('=' * 70)
print('TRAINING: Random Forest Models (extreme memory efficiency)')
print(f'  Params: {RF_PARAMS}')
print(f'  Backend: {"GPU (cuML)" if USE_GPU else "CPU (sklearn)"}')
print(f'  Strategy: Load data → train → free → load test → evaluate → free')
print('=' * 70)

clf_metrics = {}
reg_metrics = {}
clf_train_time = 0
reg_train_time = 0

for h in HORIZONS:
    print(f'\n{"─"*70}')
    print(f'HORIZON h={h}')
    print(f'{"─"*70}')
    
    # Check if already trained (skip if exists)
    clf_path = os.path.join(WEIGHTS_DIR, f'rf_classifier_h{h}.joblib')
    reg_path = os.path.join(WEIGHTS_DIR, f'rf_regressor_h{h}.joblib')
    
    h_idx = HORIZONS.index(h)
    
    # ══════════════════════════════════════════════════════════════════════
    # CLASSIFIER
    # ══════════════════════════════════════════════════════════════════════
    if os.path.exists(clf_path):
        print(f'  ✓ Classifier already exists: {clf_path}')
        print(f'    (Delete file to retrain, or it will be used for evaluation)')
    else:
        print(f'  [CLF] Loading training data into RAM...')
        X_tr = np.array(X_train_s)  # copy from mmap to RAM
        y_tr = y_cls_train[:, h_idx].astype(np.int32)
        y_tr = np.array(y_tr)       # copy to RAM
        print(f'    Train data: {X_tr.nbytes/1e9:.2f} GB  |  RSS={_mem_gb():.2f} GB')
        
        if USE_GPU:
            clf = cuRFC(
                n_estimators=RF_PARAMS['n_estimators'],
                max_depth=RF_PARAMS['max_depth'],
                min_samples_leaf=RF_PARAMS['min_samples_leaf'],
                n_streams=4,
                random_state=RF_PARAMS['random_state'],
            )
        else:
            clf = RandomForestClassifier(
                n_estimators=RF_PARAMS['n_estimators'],
                max_depth=RF_PARAMS['max_depth'],
                min_samples_leaf=RF_PARAMS['min_samples_leaf'],
                class_weight='balanced',
                n_jobs=RF_PARAMS['n_jobs'],
                random_state=RF_PARAMS['random_state'],
            )
        
        print(f'  [CLF] Training...')
        t0 = time.time()
        clf.fit(X_tr, y_tr)
        t_clf = time.time() - t0
        clf_train_time += t_clf
        
        # Free training data immediately
        del X_tr, y_tr
        gc.collect()
        print(f'    Trained in {t_clf:.1f}s  |  RSS={_mem_gb():.2f} GB')
        
        # Save immediately
        joblib.dump(clf, clf_path)
        print(f'    Saved: {clf_path}  ({os.path.getsize(clf_path)/1e6:.1f} MB)')
        
        # Free model
        del clf
        gc.collect()
        print(f'    Model freed. RSS={_mem_gb():.2f} GB')
    
    # ── Evaluate classifier ──────────────────────────────────────────────
    print(f'  [CLF] Evaluating on test set...')
    clf = joblib.load(clf_path)
    
    X_te = np.array(X_test_s)  # copy test data to RAM
    y_te = y_cls_test[:, h_idx].astype(np.int32)
    y_te = np.array(y_te)
    
    y_pred = clf.predict(X_te)
    if USE_GPU:
        y_pred = y_pred.get() if hasattr(y_pred, 'get') else np.array(y_pred)
    
    clf_metrics[f'h{h}_accuracy']        = accuracy_score(y_te, y_pred)
    clf_metrics[f'h{h}_f1_macro']        = f1_score(y_te, y_pred, average='macro', zero_division=0)
    clf_metrics[f'h{h}_f1_weighted']     = f1_score(y_te, y_pred, average='weighted', zero_division=0)
    clf_metrics[f'h{h}_precision_macro'] = precision_score(y_te, y_pred, average='macro', zero_division=0)
    clf_metrics[f'h{h}_recall_macro']    = recall_score(y_te, y_pred, average='macro', zero_division=0)
    
    print(f'    Acc={clf_metrics[f"h{h}_accuracy"]:.4f}  F1={clf_metrics[f"h{h}_f1_macro"]:.4f}')
    
    # Free everything
    del clf, X_te, y_te, y_pred
    gc.collect()
    print(f'    Eval complete, freed. RSS={_mem_gb():.2f} GB')
    
    # ══════════════════════════════════════════════════════════════════════
    # REGRESSOR
    # ══════════════════════════════════════════════════════════════════════
    if os.path.exists(reg_path):
        print(f'  ✓ Regressor already exists: {reg_path}')
        print(f'    (Delete file to retrain, or it will be used for evaluation)')
    else:
        print(f'  [REG] Loading training data into RAM...')
        X_tr = np.array(X_train_s)
        y_tr = y_reg_train[:, h_idx].astype(np.float32)
        y_tr = np.array(y_tr)
        print(f'    Train data: {X_tr.nbytes/1e9:.2f} GB  |  RSS={_mem_gb():.2f} GB')
        
        if USE_GPU:
            reg = cuRFR(
                n_estimators=RF_PARAMS['n_estimators'],
                max_depth=RF_PARAMS['max_depth'],
                min_samples_leaf=RF_PARAMS['min_samples_leaf'],
                n_streams=4,
                random_state=RF_PARAMS['random_state'],
            )
        else:
            reg = RandomForestRegressor(
                n_estimators=RF_PARAMS['n_estimators'],
                max_depth=RF_PARAMS['max_depth'],
                min_samples_leaf=RF_PARAMS['min_samples_leaf'],
                n_jobs=RF_PARAMS['n_jobs'],
                random_state=RF_PARAMS['random_state'],
            )
        
        print(f'  [REG] Training...')
        t0 = time.time()
        reg.fit(X_tr, y_tr)
        t_reg = time.time() - t0
        reg_train_time += t_reg
        
        # Free training data
        del X_tr, y_tr
        gc.collect()
        print(f'    Trained in {t_reg:.1f}s  |  RSS={_mem_gb():.2f} GB')
        
        # Save immediately
        joblib.dump(reg, reg_path)
        print(f'    Saved: {reg_path}  ({os.path.getsize(reg_path)/1e6:.1f} MB)')
        
        # Free model
        del reg
        gc.collect()
        print(f'    Model freed. RSS={_mem_gb():.2f} GB')
    
    # ── Evaluate regressor ───────────────────────────────────────────────
    print(f'  [REG] Evaluating on test set...')
    reg = joblib.load(reg_path)
    
    X_te = np.array(X_test_s)
    y_te = y_reg_test[:, h_idx].astype(np.float32)
    y_te = np.array(y_te)
    
    y_pred = reg.predict(X_te)
    if USE_GPU:
        y_pred = y_pred.get() if hasattr(y_pred, 'get') else np.array(y_pred)
    
    reg_metrics[f'h{h}_mse']  = float(mean_squared_error(y_te, y_pred))
    reg_metrics[f'h{h}_rmse'] = float(np.sqrt(reg_metrics[f'h{h}_mse']))
    reg_metrics[f'h{h}_mae']  = float(mean_absolute_error(y_te, y_pred))
    reg_metrics[f'h{h}_r2']   = float(r2_score(y_te, y_pred))
    if np.std(y_te) > 1e-12 and np.std(y_pred) > 1e-12:
        reg_metrics[f'h{h}_ic'] = float(np.corrcoef(y_te, y_pred)[0, 1])
    else:
        reg_metrics[f'h{h}_ic'] = 0.0
    
    print(f'    R²={reg_metrics[f"h{h}_r2"]:.4f}  MAE={reg_metrics[f"h{h}_mae"]:.6f}')
    
    # Free everything
    del reg, X_te, y_te, y_pred
    gc.collect()
    print(f'    Eval complete, freed. RSS={_mem_gb():.2f} GB  |  Avail={_avail_gb():.2f} GB')

# Save scaler
scaler_path = os.path.join(WEIGHTS_DIR, 'rf_scaler.joblib')
joblib.dump(scaler, scaler_path)
print(f'\n✓ Scaler saved: {scaler_path}')

print(f'\n{"═"*70}')
print(f'✅ ALL MODELS TRAINED & SAVED')
print(f'  Classifier total: {clf_train_time:.1f}s')
print(f'  Regressor total:  {reg_train_time:.1f}s')
print(f'  Models saved to:  {WEIGHTS_DIR}/')
print(f'  Peak RAM usage:   {_mem_gb():.2f} GB')
print(f'{"═"*70}')

---
## 📊 Evaluation

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 8: Print Final Metrics Summary
# ══════════════════════════════════════════════════════════════════════════════

print('=' * 70)
print('TEST SET EVALUATION SUMMARY')
print('=' * 70)

print('\n┌──────────────────────────────────────────────────────────────────┐')
print('│  RF CLASSIFIER — Test Metrics                                    │')
print('└──────────────────────────────────────────────────────────────────┘')
print_clf_metrics(clf_metrics)

print('\n┌──────────────────────────────────────────────────────────────────┐')
print('│  RF REGRESSOR — Test Metrics                                     │')
print('└──────────────────────────────────────────────────────────────────┘')
print_reg_metrics(reg_metrics)

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 9: Confusion Matrices (load models from disk one at a time)
# ══════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, len(HORIZONS), figsize=(5 * len(HORIZONS), 5))
label_names = ['DOWN', 'STAT', 'UP']

for i, h in enumerate(HORIZONS):
    # Load model from disk
    clf = joblib.load(os.path.join(WEIGHTS_DIR, f'rf_classifier_h{h}.joblib'))
    
    y_true = y_cls_test[:, i].astype(int)
    y_pred = clf.predict(X_test_s)
    if USE_GPU:
        y_pred = y_pred.get() if hasattr(y_pred, 'get') else np.array(y_pred)
    
    # Free model immediately
    del clf
    gc.collect()
    
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    
    sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
                xticklabels=label_names, yticklabels=label_names,
                ax=axes[i], vmin=0, vmax=100,
                cbar_kws={'label': '%'})
    axes[i].set_title(f'h = {h}', fontweight='bold', fontsize=13)
    axes[i].set_xlabel('Predicted')
    axes[i].set_ylabel('Actual')

plt.suptitle('RF Classifier — Confusion Matrices (% per true class)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'rf_classifier_confusion.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 10: Feature Importance (load models from disk one at a time)
# ══════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, len(HORIZONS), figsize=(6 * len(HORIZONS), 14))

for i, h in enumerate(HORIZONS):
    # ── Classifier feature importance ─────────────────────────────────────
    clf = joblib.load(os.path.join(WEIGHTS_DIR, f'rf_classifier_h{h}.joblib'))
    imp_clf = clf.feature_importances_
    if USE_GPU:
        imp_clf = imp_clf.get() if hasattr(imp_clf, 'get') else np.array(imp_clf)
    del clf
    gc.collect()
    
    sorted_idx = np.argsort(imp_clf)
    top_n = min(20, len(feat_names))
    top_idx = sorted_idx[-top_n:]
    
    axes[0, i].barh(
        [feat_names[j] for j in top_idx],
        imp_clf[top_idx],
        color='steelblue', edgecolor='none'
    )
    axes[0, i].set_title(f'Classifier h={h}', fontweight='bold')
    axes[0, i].set_xlabel('Gini Importance')
    
    # ── Regressor feature importance ──────────────────────────────────────
    reg = joblib.load(os.path.join(WEIGHTS_DIR, f'rf_regressor_h{h}.joblib'))
    imp_reg = reg.feature_importances_
    if USE_GPU:
        imp_reg = imp_reg.get() if hasattr(imp_reg, 'get') else np.array(imp_reg)
    del reg
    gc.collect()
    
    sorted_idx = np.argsort(imp_reg)
    top_idx = sorted_idx[-top_n:]
    
    axes[1, i].barh(
        [feat_names[j] for j in top_idx],
        imp_reg[top_idx],
        color='darkorange', edgecolor='none'
    )
    axes[1, i].set_title(f'Regressor h={h}', fontweight='bold')
    axes[1, i].set_xlabel('Gini Importance')

axes[0, 0].set_ylabel('CLASSIFIER\n\nFeature', fontweight='bold', fontsize=12)
axes[1, 0].set_ylabel('REGRESSOR\n\nFeature', fontweight='bold', fontsize=12)

plt.suptitle('Random Forest — Feature Importance (Top 20 per Horizon)',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'rf_feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 11: Regressor — Predicted vs Actual Scatter
# ══════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(1, len(HORIZONS), figsize=(5 * len(HORIZONS), 5))

# Sample points for plotting (avoid overplotting)
n_plot = min(5000, X_test_s.shape[0])
plot_idx = np.random.RandomState(42).choice(X_test_s.shape[0], n_plot, replace=False)
X_plot = X_test_s[plot_idx]

for i, h in enumerate(HORIZONS):
    # Load model from disk
    reg = joblib.load(os.path.join(WEIGHTS_DIR, f'rf_regressor_h{h}.joblib'))
    
    y_true = y_reg_test[plot_idx, i]
    y_pred = reg.predict(X_plot)
    if USE_GPU:
        y_pred = y_pred.get() if hasattr(y_pred, 'get') else np.array(y_pred)
    
    # Free model
    del reg
    gc.collect()
    
    axes[i].scatter(y_true, y_pred, alpha=0.15, s=5, color='teal')
    
    # Perfect prediction line
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    axes[i].plot(lims, lims, 'r--', alpha=0.8, linewidth=1.5, label='Perfect')
    
    r2 = reg_metrics[f'h{h}_r2']
    ic = reg_metrics[f'h{h}_ic']
    axes[i].set_title(f'h={h}  (R²={r2:.4f}, IC={ic:.4f})', fontweight='bold')
    axes[i].set_xlabel('Actual ΔMid')
    axes[i].set_ylabel('Predicted ΔMid')
    axes[i].legend(loc='upper left')

del X_plot
gc.collect()

plt.suptitle('RF Regressor — Predicted vs Actual Mid-Price Change', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'rf_regressor_scatter.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## 💾 Save Models & Metadata

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 12: Save Metadata (models already saved in Cell 7)
# ══════════════════════════════════════════════════════════════════════════════

print('Saving metadata...')

# ── Save metadata ────────────────────────────────────────────────────────────
metadata = {
    'timestamp': pd.Timestamp.now().isoformat(),
    'tickers': tickers,
    'horizons': HORIZONS,
    'feature_names': feat_names,
    'n_features': len(feat_names),
    'use_gpu': USE_GPU,
    'subsample': SUBSAMPLE,
    'data': {
        'n_train': int(X_train_s.shape[0]),
        'n_val':   int(X_val_s.shape[0]),
        'n_test':  int(X_test_s.shape[0]),
        'n_total': int(X_train_s.shape[0] + X_val_s.shape[0] + X_test_s.shape[0]),
    },
    'classifier': {
        'params': RF_PARAMS,
        'train_time_s': round(clf_train_time, 1),
        'test_metrics': {k: round(v, 6) for k, v in clf_metrics.items()},
    },
    'regressor': {
        'params': RF_PARAMS,
        'train_time_s': round(reg_train_time, 1),
        'test_metrics': {k: round(v, 6) for k, v in reg_metrics.items()},
    },
}

meta_path = os.path.join(WEIGHTS_DIR, 'rf_metadata.json')
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=2, default=str)
print(f'  ✓ {meta_path}')

results_meta_path = os.path.join(RESULTS_DIR, 'rf_results.json')
with open(results_meta_path, 'w') as f:
    json.dump(metadata, f, indent=2, default=str)
print(f'  ✓ {results_meta_path}')

print('\n✅ Metadata saved.')

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# Cell 13: Final Summary
# ══════════════════════════════════════════════════════════════════════════════

print('\n' + '=' * 80)
print('FINAL RESULTS SUMMARY')
print('=' * 80)

print('\n┌─────────────────────────────────────────────────────────────────────┐')
print('│  MODEL 1: RF CLASSIFIER  (Label: DOWN=0, STAT=1, UP=2)             │')
print('│  Input: 38 features (OFI + Micro + Raw LOB)                         │')
print('│  Output: Direction class per horizon                                │')
print('└─────────────────────────────────────────────────────────────────────┘')
print_clf_metrics(clf_metrics)

print('\n┌─────────────────────────────────────────────────────────────────────┐')
print('│  MODEL 2: RF REGRESSOR  (Target: ΔMid = mid_{t+h} − mid_t)         │')
print('│  Input: 38 features (OFI + Micro + Raw LOB)                         │')
print('│  Output: Continuous mid-price change per horizon                    │')
print('└─────────────────────────────────────────────────────────────────────┘')
print_reg_metrics(reg_metrics)

# ── OFI importance summary (load one model briefly) ───────────────────────────
print('\n' + '=' * 80)
print('OFI FEATURE IMPORTANCE ANALYSIS')
print('=' * 80)

ofi_features = [f for f in feat_names if f.startswith('ofi_')]
micro_features = ['mid_price', 'spread', 'volume_imbalance',
                  'weighted_mid_price', 'bid_depth', 'ask_depth',
                  'total_depth', 'depth_imbalance']
lob_features = [f for f in feat_names if f not in ofi_features and f not in micro_features]

ofi_idx   = [feat_names.index(f) for f in ofi_features if f in feat_names]
micro_idx = [feat_names.index(f) for f in micro_features if f in feat_names]
lob_idx   = [feat_names.index(f) for f in lob_features if f in feat_names]

for h in HORIZONS:
    clf = joblib.load(os.path.join(WEIGHTS_DIR, f'rf_classifier_h{h}.joblib'))
    imp = clf.feature_importances_
    if USE_GPU:
        imp = imp.get() if hasattr(imp, 'get') else np.array(imp)
    del clf
    gc.collect()
    
    ofi_imp   = imp[ofi_idx].sum()   if ofi_idx   else 0
    micro_imp = imp[micro_idx].sum() if micro_idx else 0
    lob_imp   = imp[lob_idx].sum()   if lob_idx   else 0
    total_imp = ofi_imp + micro_imp + lob_imp
    
    print(f'  h={h:>3d}  |  '
          f'OFI: {ofi_imp/total_imp*100:5.1f}%  '
          f'Micro: {micro_imp/total_imp*100:5.1f}%  '
          f'Raw LOB: {lob_imp/total_imp*100:5.1f}%')

print('\n' + '=' * 80)
print('Files saved:')
print('  model_weights/rf_classifier_h{10,20,50,100}.joblib')
print('  model_weights/rf_regressor_h{10,20,50,100}.joblib')
print('  model_weights/rf_scaler.joblib')
print('  model_weights/rf_metadata.json')
print('  results/rf_results.json')
print('  results/rf_classifier_confusion.png')
print('  results/rf_feature_importance.png')
print('  results/rf_regressor_scatter.png')
print('=' * 80)